# Model Development and Validation

Machine learning models were developed using molecular descriptors, Morgan fingerprints, and combined feature representations to predict anti-leishmanial activity (pIC50).

In [ ]:
import pandas as pd

X_desc = pd.read_csv("../data/processed/X_descriptors.csv")
X_fp = pd.read_csv("../data/processed/X_fingerprints.csv")
X_combined = pd.read_csv("../data/processed/X_combined.csv")
y = pd.read_csv("../data/processed/y_pIC50.csv")

print(X_desc.shape)
print(X_fp.shape)
print(X_combined.shape)
print(y.shape)

## Train-Test Split

The dataset was divided into training and testing sets. The training set was used for model development, while the testing set was reserved for independent performance evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_desc,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

## Random Forest Regression

A Random Forest Regressor was trained using molecular descriptors as input features. Random Forest was selected because it can model complex non-linear relationships between molecular properties and biological activity.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train.values.ravel())

In [ ]:
y_pred = rf.predict(X_test)

## Model Evaluation

Model performance was assessed using the coefficient of determination (R²), mean absolute error (MAE), and root mean squared error (RMSE).

In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("R²:", round(r2, 3))
print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))

## Cross-Validation

Five-fold cross-validation was performed to assess model robustness and estimate predictive performance across different subsets of the dataset.

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    rf,
    X_desc,
    y.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=1
)

print("CV Scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)


X_train, X_test, y_train, y_test = train_test_split(
    X_fp,
    y,
    test_size=0.2,
    random_state=42
)


rf_fp = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=1
)

rf_fp.fit(
    X_train,
    y_train.values.ravel()
)


y_pred = rf_fp.predict(X_test)


r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)


cv_scores = cross_val_score(
    rf_fp,
    X_fp,
    y.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=1
)

print("R²:", round(r2, 3))
print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("CV Scores:", cv_scores)
print("Mean CV R²:", round(cv_scores.mean(), 3))

Morgan fingerprints substantially outperformed traditional molecular descriptors in predicting anti-leishmanial activity. This suggests that structural subfeatures and local atomic environments contribute more strongly to biological activity than global physicochemical descriptors alone.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_combined,
    y,
    test_size=0.2,
    random_state=42
)

# Random Forest model
rf_combined = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=1
)

# Train model
rf_combined.fit(
    X_train,
    y_train.values.ravel()
)

# Predictions
y_pred = rf_combined.predict(X_test)

# Evaluation metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("="*50)
print("TEST SET PERFORMANCE")
print("="*50)
print(f"R²   : {r2:.3f}")
print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")

# 5-fold Cross Validation
cv_scores = cross_val_score(
    rf_combined,
    X_combined,
    y.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=1
)

print("\n" + "="*50)
print("5-FOLD CROSS VALIDATION")
print("="*50)
print("CV Scores:", cv_scores)
print(f"Mean CV R² : {cv_scores.mean():.3f}")
print(f"Std CV R²  : {cv_scores.std():.3f}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# Random Forest base model
rf = RandomForestRegressor(random_state=42, n_jobs=1)

# Hyperparameter search space
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30, 50],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.3, 0.5, None]
}

# Randomized search
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=25,
    cv=3,
    scoring="r2",
    random_state=42,
    n_jobs=1,
    verbose=1
)

random_search.fit(X_train, y_train.values.ravel())

print("Best Parameters:", random_search.best_params_)
print("Best CV R²:", round(random_search.best_score_, 3))

In [ ]:
# Best tuned model
best_rf = random_search.best_estimator_

# Predictions on test set
y_pred_tuned = best_rf.predict(X_test)

# Metrics
r2_tuned = r2_score(y_test, y_pred_tuned)
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = root_mean_squared_error(y_test, y_pred_tuned)

print("="*50)
print("TUNED RANDOM FOREST TEST PERFORMANCE")
print("="*50)
print(f"R²   : {r2_tuned:.3f}")
print(f"MAE  : {mae_tuned:.3f}")
print(f"RMSE : {rmse_tuned:.3f}")

In [ ]:
print("="*50)
print("BASELINE vs TUNED RANDOM FOREST")
print("="*50)
print(f"Baseline Test R² : {r2:.3f}")
print(f"Tuned Test R²    : {r2_tuned:.3f}")
print(f"Baseline MAE     : {mae:.3f}")
print(f"Tuned MAE        : {mae_tuned:.3f}")
print(f"Baseline RMSE    : {rmse:.3f}")
print(f"Tuned RMSE       : {rmse_tuned:.3f}")

## Hyperparameter Optimization of Random Forest

RandomizedSearchCV was used to optimize Random Forest hyperparameters. The tuned model achieved a modest improvement over the baseline model, suggesting that feature representation had a greater impact on predictive performance than parameter optimization.

## XGBoost Regression

An Extreme Gradient Boosting (XGBoost) model was trained using the combined descriptor-fingerprint representation. XGBoost is a powerful ensemble learning algorithm capable of capturing complex non-linear relationships and feature interactions.

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_combined,
    y,
    test_size=0.2,
    random_state=42
)

# XGBoost model
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1
)

# Train
xgb.fit(
    X_train,
    y_train.values.ravel()
)

# Predict
y_pred_xgb = xgb.predict(X_test)

# Metrics
r2_xgb = r2_score(y_test, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = root_mean_squared_error(y_test, y_pred_xgb)

print("="*50)
print("XGBOOST TEST PERFORMANCE")
print("="*50)
print(f"R²   : {r2_xgb:.3f}")
print(f"MAE  : {mae_xgb:.3f}")
print(f"RMSE : {rmse_xgb:.3f}")

In [ ]:
cv_scores_xgb = cross_val_score(
    xgb,
    X_combined,
    y.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=1
)

print("CV Scores:", cv_scores_xgb)
print("Mean CV R²:", round(cv_scores_xgb.mean(), 3))
print("Std CV R² :", round(cv_scores_xgb.std(), 3))

In [ ]:
print("="*60)
print("MODEL COMPARISON")
print("="*60)

print(f"Random Forest R² : {r2_tuned:.3f}")
print(f"XGBoost R²       : {r2_xgb:.3f}")

print(f"Random Forest RMSE : {rmse_tuned:.3f}")
print(f"XGBoost RMSE       : {rmse_xgb:.3f}")

While the tuned Random Forest model achieved the highest test-set performance, XGBoost produced the highest cross-validated R² score, suggesting greater robustness and generalization across different data partitions.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

extra = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=1
)

extra.fit(
    X_train,
    y_train.values.ravel()
)

y_pred_extra = extra.predict(X_test)

r2_extra = r2_score(y_test, y_pred_extra)
mae_extra = mean_absolute_error(y_test, y_pred_extra)
rmse_extra = root_mean_squared_error(y_test, y_pred_extra)

cv_scores_extra = cross_val_score(
    extra,
    X_combined,
    y.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=1
)

print("R²:", round(r2_extra, 3))
print("MAE:", round(mae_extra, 3))
print("RMSE:", round(rmse_extra, 3))
print("Mean CV R²:", round(cv_scores_extra.mean(), 3))
print("Std CV R²:", round(cv_scores_extra.std(), 3))

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

cat = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=0,
    random_state=42
)

cat.fit(
    X_train,
    y_train.values.ravel()
)

y_pred_cat = cat.predict(X_test)

r2_cat = r2_score(y_test, y_pred_cat)
mae_cat = mean_absolute_error(y_test, y_pred_cat)
rmse_cat = root_mean_squared_error(y_test, y_pred_cat)

cv_scores_cat = cross_val_score(
    cat,
    X_combined,
    y.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=1
)

print("R²:", round(r2_cat, 3))
print("MAE:", round(mae_cat, 3))
print("RMSE:", round(rmse_cat, 3))
print("Mean CV R²:", round(cv_scores_cat.mean(), 3))
print("Std CV R²:", round(cv_scores_cat.std(), 3))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "RF Descriptors",
        "RF Fingerprints",
        "RF Combined",
        "RF Tuned",
        "XGBoost",
        "Extra Trees",
        "CatBoost"
    ],
    "Test_R2": [
        0.471,
        0.613,
        0.616,
        0.623,
        0.618,
        0.622,
        0.542
    ]
})

results